<a href="https://colab.research.google.com/github/halimAhtasham/DeepLearning/blob/main/07_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 7: Hyperparameter tuning**

### **Objective**

Optimize the baseline machine learning models using cross-validation
and systematic hyperparameter search.

Models:
1. Logistic Regression
2. K-Nearest Neighbors
3. Decision Tree
4. Random Forest

Methods:
- GridSearchCV
- Stratified 5-Fold Cross-Validation

The final test set will remain untouched during hyperparameter tuning.

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_PATH = "/content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction"

DATASET_PATH = PROJECT_PATH + "/Dataset"
RESULT_PATH = PROJECT_PATH + "/Results"
MODEL_PATH = PROJECT_PATH + "/Models"

## **Dataset Load**

In [4]:
df = pd.read_csv(DATASET_PATH + "/heart.csv")

df_clean = df.drop_duplicates().copy()

print("Original dataset:", df.shape)
print("Cleaned dataset:", df_clean.shape)

Original dataset: (1025, 14)
Cleaned dataset: (302, 14)


## **Separate Features**

In [5]:
X= df_clean.drop("target", axis=1)
y=df_clean["target"]

## **Train / Test Split**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
print("Training:", X_train.shape)
print("Test:", X_test.shape)

Training: (241, 13)
Test: (61, 13)


## **Stratified 5-Fold Cross Validation**

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# **Logistic Regression Hyperparameter Tuning**

In [10]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=2000))
])

In [11]:
logistic_params = {
    "model__C":[0.01,0.1,1,10,100]
}

In [12]:
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=False
)

In [13]:
logistic_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1, param_grid={'model__C': [0.01, 0.1, 1, 10, 100]},
             scoring='f1')

In [14]:
print("Best Parameters:")
print(logistic_grid.best_params_)

Best Parameters:
{'model__C': 1}


In [15]:
print("Best CV F1:")
print(logistic_grid.best_score_)

Best CV F1:
0.8600187148574244


In [16]:
best_logistic = logistic_grid.best_estimator_

In [17]:
logistic_results = pd.DataFrame(
    logistic_grid.cv_results_
)

logistic_results[
    [
        "param_model__C",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

,param_model__C,mean_test_score,std_test_score,rank_test_score
2,1.00,0.860019,0.068267,1
3,10.00,0.854953,0.062324,2
4,100.00,0.854953,0.062324,2
1,0.10,0.854147,0.063010,4
0,0.01,0.848444,0.054938,5
